<a href="https://colab.research.google.com/github/sw206/Choose-your-path-game/blob/main/Neural_Network_for_Tomato_Disease_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Setting Up the Environment & Extracting the Dataset

Since image datasets can get huge, the PlantVillage tomato images were zipped up into a `.7z` file and kept in Google Drive.

To work with them in Colab without lag, we need to:
1. **Mount Google Drive** so Colab can find our zipped file.
2. **Unzip the data (`py7zr`)** into Colab's local directory (`/content/extracted_data`). Reading images directly from Colab's local memory is much faster when we start preprocessing and training later on.

In [ ]:
# Environment Configuration, Google Drive Mount & Dataset Extraction
import os
import shutil
import glob
from google.colab import drive
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import pandas as pd

# Mount Google Drive to access stored files
drive.mount('/content/drive')

# Install 7zip extraction tool
!pip install py7zr -q

import py7zr

# Path to your uploaded data.7z file in Drive
archive_path = '/content/drive/MyDrive/Preprocesseddata.7z'
extracted_path = '/content/extracted_data'

print("Extracting dataset...")
with py7zr.SevenZipFile(archive_path, mode='r') as z:
    z.extractall(path=extracted_path)
print("Extraction complete!")

# 2. Defining Target Classes, File Indexing, and Class Statistics

###  Updating to All 10 Target Classes
We updated our pipeline to include all 10 available tomato categories from the PlantVillage dataset rather than filtering down to a subset:
* **9 Disease Conditions**: `Bacterial_spot`, `Early_blight`, `Late_blight`, `Leaf_Mold`, `Septoria_leaf_spot`, `Spider_mites Two-spotted_spider_mite`, `Target_Spot`, `Tomato_Yellow_Leaf_Curl_Virus`, and `Tomato_mosaic_virus`.
* **1 Control Group**: `Healthy`.

###  Image Indexing & Distribution Counting
Before splitting the dataset or feeding images to our model, we need to map where every file is located and check the sample counts across all 10 categories:
* **`find_image_dirs()`**: Recursively scans the unzipped folder for standard image extensions (`.png`, `.jpg`, `.jpeg`).
* **Path Matching**: Matches file paths to our class list, handling space-to-underscore formatting to ensure all images are accurately mapped.
* **Summary Table**: Converts the final counts into a Pandas DataFrame for a clean breakdown of our dataset's balance.

In [ ]:
# ==============================================================================
# STEP 2: Target Classes, Robust File Indexing & Quantitative EDA
# ==============================================================================

import os
import pandas as pd

# 1. Define Standardized 10 Tomato Target Class Labels
SELECTED_CLASSES = [
    'Bacterial_spot',
    'Early_blight',
    'Late_blight',
    'Leaf_Mold',
    'Septoria_leaf_spot',
    'Spider_mites_Two-spotted_spider_mite',
    'Target_Spot',
    'Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato_mosaic_virus',
    'Healthy'
]

# 2. Function to search and index images by matching class names
def find_image_dirs(base_path):
    all_files = []
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                all_files.append(os.path.join(root, file))
    return all_files

all_images = find_image_dirs(extracted_path)
print(f"Total images found in archive: {len(all_images)}")

# 3. Flexible Class Distribution Counting Logic
class_image_paths = {c: [] for c in SELECTED_CLASSES}

for img_path in all_images:
    path_lower = img_path.lower()

    # Special check for Spider Mites to capture common variations (spider, mite)
    if 'spider' in path_lower or 'mite' in path_lower:
        class_image_paths['Spider_mites_Two-spotted_spider_mite'].append(img_path)
        continue

    # Standard matching for the remaining 9 classes
    for cls in SELECTED_CLASSES:
        if cls != 'Spider_mites_Two-spotted_spider_mite':
            # Clean folder string for matching
            cls_clean = cls.lower().replace('_', '')
            path_clean = path_lower.replace('_', '').replace(' ', '')
            if cls_clean in path_clean:
                class_image_paths[cls].append(img_path)
                break

# 4. Display Class Statistics Table
class_counts = {cls: len(paths) for cls, paths in class_image_paths.items()}
df_stats = pd.DataFrame(list(class_counts.items()), columns=['Class Name', 'Total Images'])

print("\n--- Selected Target Classes Statistics ---")
print(df_stats.to_string(index=False))

# 3. Dataset Visualizations & Exploratory Inspection

### Class Balance Analysis (Bar Plot)
To evaluate sample distributions across all 10 categories, we plot total image counts using `seaborn` and `matplotlib`.
* Visualizing the class breakdown helps us spot any severe class imbalance before we train our model.
* We rotate the x-axis labels to prevent text overlap now that we are plotting all 10 classes.

###  Qualitative Visual Inspection (Sample Image Grid)
Checking the actual leaf images helps us confirm that the images are uncorrupted and observe physical symptoms (e.g., leaf mold, spots, curling, vs. clean foliage) across categories.
* **Metadata Verification**: Displays image dimensions (`img.size`) and color channel mode (`img.mode`) to ensure compatibility with standard convolutional layers.

In [ ]:
# ==============================================================================
# STEP 3: Exploratory Data Analysis — Visualizations & Sample Inspection
# ==============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# 1. Plot Class Distribution Bar Chart (Updated for 10 Classes)
plt.figure(figsize=(12, 6))
ax = sns.barplot(x='Class Name', y='Total Images', data=df_stats, palette='viridis')
plt.title('Exploratory Data Analysis: Image Distribution Across All 10 Classes', fontsize=14, fontweight='bold')
plt.xlabel('Tomato Leaf Class', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)

# Rotate x-axis labels 45 degrees to fit all 10 class names cleanly
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Display exact counts on top of each bar for clarity
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=10)

plt.tight_layout()
plt.show()

# 2. Qualitative Visual Inspection Grid (1 Image Per Class)
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

print("\n--- Image Metadata Sample ---")
for idx, cls in enumerate(SELECTED_CLASSES):
    if class_image_paths[cls]:
        sample_path = class_image_paths[cls][0]
        img = Image.open(sample_path)

        # Display image metadata for the first class as a reference
        if idx == 0:
            print(f"Sample Resolution: {img.size} (Width x Height)")
            print(f"Color Mode: {img.mode}")

        axes[idx].imshow(img)
        axes[idx].set_title(cls.replace('_', ' '), fontsize=10, fontweight='bold')
        axes[idx].axis('off')

plt.suptitle('Representative Sample Images Across 10 Target Classes', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# 4. Dataset Partitioning (70% Train / 15% Validation / 15% Test)

### Partitioning Strategy & Reproducibility
To properly evaluate our neural network and prevent data leakage, we split our dataset into three distinct sets:
* **Train Set (70%)**: Used by the model to learn weights during training.
* **Validation Set (15%)**: Used during training to monitor performance, tune hyper-parameters, and prevent overfitting.
* **Test Set (15%)**: Held-out evaluation set used only for final model performance reporting.

We set a fixed random seed (`SEED = 42`) prior to shuffling so that every team member gets the exact same file splits.

### Directory Structuring
We automatically organize the partitioned files into a structured directory hierarchy:
```text
dataset_split/
├── train/
│   ├── Bacterial_spot/
│   ├── Early_blight/
│   └── ... (all 10 classes)
├── val/
│   └── ... (all 10 classes)
└── test/
    └── ... (all 10 classes)

In [ ]:
# ==============================================================================
# STEP 4: Dataset Partitioning (70% Train, 15% Validation, 15% Test)
# ==============================================================================

import os
import shutil
import random
import pandas as pd

# 1. Configuration & Reproducibility
SEED = 42
random.seed(SEED)

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Output directory for structured splits
OUTPUT_BASE_DIR = '/content/dataset_split'
SPLITS = ['train', 'val', 'test']

# Clear previous splits if re-running cell
if os.path.exists(OUTPUT_BASE_DIR):
    shutil.rmtree(OUTPUT_BASE_DIR)

# 2. Create Directory Structure for All 10 Classes
for split in SPLITS:
    for cls in SELECTED_CLASSES:
        os.makedirs(os.path.join(OUTPUT_BASE_DIR, split, cls), exist_ok=True)

# 3. Partition Files Per Class
split_summary = {cls: {'train': 0, 'val': 0, 'test': 0} for cls in SELECTED_CLASSES}

for cls in SELECTED_CLASSES:
    paths = class_image_paths[cls].copy()
    random.shuffle(paths)  # Shuffle to ensure randomized distribution

    total_imgs = len(paths)
    if total_imgs == 0:
        continue

    train_end = int(total_imgs * TRAIN_RATIO)
    val_end = train_end + int(total_imgs * VAL_RATIO)

    train_paths = paths[:train_end]
    val_paths = paths[train_end:val_end]
    test_paths = paths[val_end:]

    # Move files to respective folders
    for path in train_paths:
        shutil.copy(path, os.path.join(OUTPUT_BASE_DIR, 'train', cls, os.path.basename(path)))
    for path in val_paths:
        shutil.copy(path, os.path.join(OUTPUT_BASE_DIR, 'val', cls, os.path.basename(path)))
    for path in test_paths:
        shutil.copy(path, os.path.join(OUTPUT_BASE_DIR, 'test', cls, os.path.basename(path)))

    # Record split metrics
    split_summary[cls]['train'] = len(train_paths)
    split_summary[cls]['val'] = len(val_paths)
    split_summary[cls]['test'] = len(test_paths)

# 4. Display Partition Summary Table
df_splits = pd.DataFrame.from_dict(split_summary, orient='index')
df_splits['Total'] = df_splits.sum(axis=1)

print("\n--- Dataset Partition Summary (10 Classes) ---")
print(df_splits.to_string())
print(f"\nSuccessfully structured split dataset at: {OUTPUT_BASE_DIR}")

# 5. PyTorch Transformations & DataLoader Pipeline

### Image Preprocessing & Transfer Learning Normalization
To prepare our raw images for convolutional backbones (e.g., ResNet, EfficientNet, or MobileNet), we define standardized PyTorch vision transformations via `torchvision.transforms`:
* **Resizing (224 × 224)**: Standard input resolution required by pretrained transfer learning architectures.
* **Tensor Conversion (`ToTensor`)**: Converts PIL image range $[0, 255]$ to floating-point tensors scaled to $[0.0, 1.0]$.
* **ImageNet Normalization**: Scales pixel values using standard ImageNet mean ($\mu = [0.485, 0.456, 0.406]$) and standard deviation ($\sigma = [0.229, 0.224, 0.225]$) to leverage pretrained weights effectively.

### 5.2 Training Set Data Augmentation
To combat overfitting and enhance generalizability across varied field conditions, we apply stochastic data augmentations **only to the training split**:
* **`RandomHorizontalFlip(p=0.5)`**: Simulates spatial symmetry and orientation variations.
* **`RandomRotation(degrees=15)`**: Mimics minor camera tilt and leaf angle differences.
* **`ColorJitter`**: Slightly varies brightness and contrast to simulate changing outdoor lighting conditions.

*Note: Validation and testing splits strictly use deterministic resizing and normalization without random augmentations to ensure reproducible evaluation.*

### 5.3 Batch DataLoader Instantiation
We instantiate PyTorch `ImageFolder` dataset abstractions and encapsulate them in `DataLoader` instances with:
* `batch_size = 32`: Efficient hardware utilization for GPU acceleration.
* `shuffle = True`: Enabled for training to prevent batch order bias across epochs.
* `num_workers = 2`: Parallelized data loading for faster execution in Google Colab.

In [ ]:
# ==============================================================================
# STEP 5: PyTorch Transformations, ImageFolder Datasets, & DataLoaders
# ==============================================================================

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Global Configurations
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
NUM_WORKERS = 2  # Colab standard worker allocation

# ImageNet standard normalization statistics
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# 2. Define Transformations
# Training Pipeline: Includes Augmentations + Normalization
train_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Validation & Testing Pipeline: Deterministic Normalization Only
eval_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# 3. Build ImageFolder Datasets
DATA_DIR = '/content/dataset_split'

image_datasets = {
    'train': datasets.ImageFolder(root=f"{DATA_DIR}/train", transform=train_transforms),
    'val':   datasets.ImageFolder(root=f"{DATA_DIR}/val",   transform=eval_transforms),
    'test':  datasets.ImageFolder(root=f"{DATA_DIR}/test",  transform=eval_transforms)
}

# 4. Construct DataLoaders
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True),
    'val':   DataLoader(image_datasets['val'],   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True),
    'test':  DataLoader(image_datasets['test'],  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
}

# 5. Verification & Batch Sanity Check
print("\n--- DataLoader Initialization Summary ---")
for split in ['train', 'val', 'test']:
    dataset_len = len(image_datasets[split])
    loader_len  = len(dataloaders[split])
    print(f" • {split.capitalize():<6} Dataset: {dataset_len:>5} images | {loader_len:>3} batches (batch size = {BATCH_SIZE})")

# Print target class to index mappings
print("\n--- PyTorch Class-to-Index Mapping (10 Target Classes) ---")
for class_name, idx in image_datasets['train'].class_to_idx.items():
    print(f" Index {idx}: {class_name}")

# Inspect a single batch tensor shape
sample_images, sample_labels = next(iter(dataloaders['train']))
print(f"\nSample Batch Tensor Shape: {sample_images.shape} (Batch, Channels, Height, Width)")
print(f"Sample Batch Labels Shape: {sample_labels.shape}")

## 6. Neural Network Model Definition

For our image classification task, we'll use a pre-trained **ResNet18** model from `torchvision.models`. Using a pre-trained model allows us to leverage features learned from a large dataset like ImageNet, which is highly effective for transfer learning in computer vision tasks. We'll modify the final fully connected layer to match our 10 target classes.

In [ ]:
# ==============================================================================
# STEP 6: Neural Network Model Definition (ResNet18 for Transfer Learning)
# ==============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# 1. Device Configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load Pre-trained ResNet18 Model
resnet_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT) # Use default pre-trained weights

# Freeze all parameters in the network
for param in resnet_model.parameters():
    param.requires_grad = False

# 3. Modify the final classification layer for our 10 classes
num_ftrs_resnet = resnet_model.fc.in_features # Get the number of features entering the last layer
resnet_model.fc = nn.Linear(num_ftrs_resnet, len(SELECTED_CLASSES)) # Replace the final layer

# 4. Move model to the selected device
resnet_model = resnet_model.to(device)

print("\n--- ResNet18 Model Architecture ---")
print(resnet_model)


## 7. Model Compilation (Loss Function, Optimizer & Learning Rate Scheduler)

To prepare our model for training, we need to define the following:

*   **Loss Function**: We'll use `CrossEntropyLoss`, which is standard for multi-class classification.
*   **Optimizer**: We'll use `Adam` to adjust model weights during training.
*   **Learning Rate Scheduler**: A `StepLR` scheduler will reduce the learning rate by a factor of 0.1 every 7 epochs, helping the model converge more effectively.

In [ ]:
# ==============================================================================
# STEP 7: Model Compilation (Loss Function Only)
# ==============================================================================

# 1. Define Loss Function
criterion = nn.CrossEntropyLoss()

print("\n--- Model Compilation Summary ---")
print(f"Loss Function: {criterion}")
print("Optimizer and Scheduler will be defined dynamically for each model before training.")


### Alternative Model: EfficientNet_B0

**EfficientNet_B0** is a highly efficient convolutional neural network that scales its depth, width, and resolution systematically. It offers a good balance of performance and computational cost, making it suitable for many image classification tasks. We'll load its pretrained weights and adapt its classifier for our 10 classes.

In [ ]:
# ==============================================================================
# STEP 6.1: Define EfficientNet_B0 Model
# ==============================================================================

efficientnet_model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
for param in efficientnet_model.parameters():
    param.requires_grad = False

num_ftrs_efficientnet = efficientnet_model.classifier[1].in_features
efficientnet_model.classifier[1] = nn.Linear(num_ftrs_efficientnet, len(SELECTED_CLASSES))
efficientnet_model = efficientnet_model.to(device)

print("\n--- EfficientNet_B0 Model Architecture ---")
print(efficientnet_model)


### Alternative Model: ConvNeXt_Tiny

**ConvNeXt_Tiny** is a modern convolutional network architecture that re-examines the design of Vision Transformers and applies similar design principles to a purely convolutional model. It often achieves state-of-the-art performance while maintaining the inductive biases of CNNs. We'll utilize its pretrained weights and customize its final layer.

In [ ]:
# ==============================================================================
# STEP 6.2: Define ConvNeXt_Tiny Model
# ==============================================================================

convnext_model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
for param in convnext_model.parameters():
    param.requires_grad = False

num_ftrs_convnext = convnext_model.classifier[2].in_features
convnext_model.classifier[2] = nn.Linear(num_ftrs_convnext, len(SELECTED_CLASSES))
convnext_model = convnext_model.to(device)

print("\n--- ConvNeXt_Tiny Model Architecture ---")
print(convnext_model)


### How to Select a Model

To use one of the alternative models (VGG16, MobileNetV2, EfficientNet_B0, or ConvNeXt_Tiny), you need to:

1.  **Comment out** the `model = models.resnet18(...)` block in the 'STEP 6: Neural Network Model Definition' cell.
2.  **Uncomment** the desired model block (e.g., `model_efficientnet = models.efficientnet_b0(...)`) in its respective cell.
3.  **Assign the chosen model** to the `model` variable for subsequent steps (e.g., `model = model_efficientnet`).

For example, to use EfficientNet_B0, your code in 'STEP 6' and the EfficientNet_B0 block would look something like this:

```python
# (in STEP 6 cell)
# model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
# ...

# (in EfficientNet_B0 cell, uncomment these lines and ensure 'model' is assigned)
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT) # Uncomment this line
for param in model.parameters():
    param.requires_grad = False

num_ftrs = model.classifier[1].in_features # Note the index for classifier layer
model.classifier[1] = nn.Linear(num_ftrs, len(SELECTED_CLASSES))
model = model.to(device)
print("\n--- EfficientNet_B0 Model Architecture ---")
print(model)
```


## 8. Model Training

Now we'll implement the training loop. The training process involves iterating over the dataset multiple times (epochs), performing forward and backward passes, and updating model weights. We'll also track training and validation loss and accuracy to monitor performance and save the model with the best validation accuracy.

In [ ]:
# ==============================================================================
# STEP 8: Model Training (Training Loop for Multiple Models)
# ==============================================================================

import time
import copy

NUM_EPOCHS = 25 # You can adjust this

def train_model(model, criterion, optimizer, scheduler, num_epochs=NUM_EPOCHS):
    """
    Trains a given model and returns the best model weights and training history.
    """
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward
                # track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # deep copy the model if it's the best validation accuracy
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')

    # Load best model weights for the returned model
    model.load_state_dict(best_model_wts)
    return model, history, best_acc

# ==============================================================================
# Train All Models Sequentially
# ==============================================================================

# Define a dictionary of models to train
models_to_train = {
    'ResNet18': resnet_model,
    'EfficientNet_B0': efficientnet_model,
    'ConvNeXt_Tiny': convnext_model
}

# Store training results for each model
all_trained_models_results = {}

for model_name, current_model in models_to_train.items():
    print(f"\n--- Starting Training for {model_name} ---")

    # Define optimizer and scheduler specific to the current model
    # Need to handle different parameter names for the final layer of each model
    params_to_optimize = None
    if model_name == 'ResNet18':
        params_to_optimize = current_model.fc.parameters()
    elif model_name == 'EfficientNet_B0':
        params_to_optimize = current_model.classifier[1].parameters()
    elif model_name == 'ConvNeXt_Tiny':
        params_to_optimize = current_model.classifier[2].parameters()
    else:
        raise ValueError(f"Unknown model type: {model_name}")

    optimizer = optim.Adam(params_to_optimize, lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    # Train the model
    trained_model, history, best_val_acc = train_model(current_model, criterion, optimizer, scheduler, num_epochs=NUM_EPOCHS)

    # Store results
    all_trained_models_results[model_name] = {
        'model': trained_model,
        'history': history,
        'best_val_acc': best_val_acc
    }

    # Save the best model for this specific architecture
    model_save_path_specific = f'/content/best_model_{model_name.lower().replace("_", "")}.pth'
    torch.save(trained_model.state_dict(), model_save_path_specific)
    print(f"Best {model_name} model saved to {model_save_path_specific}")

print("\n--- All Models Training Complete ---")



--- Starting Training for ResNet18 ---
Epoch 1/25
----------
train Loss: 0.6014 Acc: 0.8233
val Loss: 0.4257 Acc: 0.8740

Epoch 2/25
----------
train Loss: 0.4462 Acc: 0.8599
val Loss: 0.3614 Acc: 0.8860

Epoch 3/25
----------
train Loss: 0.3896 Acc: 0.8779
val Loss: 0.3160 Acc: 0.8975

Epoch 4/25
----------
train Loss: 0.3630 Acc: 0.8859


## 9. Visualize Training History for All Trained Models

To understand how our models performed during training, we'll plot the training and validation accuracy and loss over epochs for each trained model. This helps us identify overfitting or underfitting and assess the models' learning progress.

In [ ]:
# ==============================================================================
# STEP 9: Visualize Training History for All Trained Models
# ==============================================================================

import matplotlib.pyplot as plt

# Plotting training and validation accuracy for all models
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
for model_name, results in all_trained_models_results.items():
    history = results['history']
    plt.plot(history['train_acc'], label=f'{model_name} Train Acc')
    plt.plot(history['val_acc'], label=f'{model_name} Val Acc', linestyle='--')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

# Plotting training and validation loss for all models
plt.subplot(1, 2, 2)
for model_name, results in all_trained_models_results.items():
    history = results['history']
    plt.plot(history['train_loss'], label=f'{model_name} Train Loss')
    plt.plot(history['val_loss'], label=f'{model_name} Val Loss', linestyle='--')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# STEP 10: Import Evaluation Libraries
# ==============================================================================

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import label_binarize

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    auc
)

In [ ]:
# ==============================================================================
# STEP 11: Evaluate All Trained Models
# ==============================================================================

NUM_CLASSES = len(SELECTED_CLASSES)

evaluation_results = {}

all_predictions = {}

for model_name, results in all_trained_models_results.items():

    print("="*80)
    print(f"Evaluating {model_name}")
    print("="*80)

    model = results["model"]
    model.eval()

    predictions = []
    labels = []
    probabilities = []

    with torch.no_grad():

        for images, target in dataloaders["test"]:

            images = images.to(device)
            target = target.to(device)

            outputs = model(images)

            probs = torch.softmax(outputs, dim=1)

            _, preds = torch.max(outputs,1)

            predictions.extend(preds.cpu().numpy())
            labels.extend(target.cpu().numpy())
            probabilities.extend(probs.cpu().numpy())

    predictions = np.array(predictions)
    labels = np.array(labels)
    probabilities = np.array(probabilities)

    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='weighted')
    recall = recall_score(labels, predictions, average='weighted')
    f1 = f1_score(labels, predictions, average='weighted')

    labels_bin = label_binarize(labels, classes=np.arange(NUM_CLASSES))

    auc_score = roc_auc_score(
        labels_bin,
        probabilities,
        multi_class='ovr',
        average='weighted'
    )

    evaluation_results[model_name] = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "AUC": auc_score
    }

    all_predictions[model_name] = {
        "labels": labels,
        "predictions": predictions,
        "probabilities": probabilities
    }

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"AUC      : {auc_score:.4f}")

    print("\nClassification Report\n")

    print(classification_report(
        labels,
        predictions,
        target_names=SELECTED_CLASSES
    ))

In [ ]:
# ==============================================================================
# STEP 12: Confusion Matrices
# ==============================================================================

for model_name in all_predictions.keys():

    cm = confusion_matrix(
        all_predictions[model_name]["labels"],
        all_predictions[model_name]["predictions"]
    )

    plt.figure(figsize=(10,8))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=SELECTED_CLASSES,
        yticklabels=SELECTED_CLASSES
    )

    plt.title(f"{model_name} Confusion Matrix")

    plt.xlabel("Predicted")

    plt.ylabel("Actual")

    plt.xticks(rotation=90)

    plt.yticks(rotation=0)

    plt.tight_layout()

    plt.show()

In [ ]:
# ==============================================================================
# STEP 13: ROC Curves
# ==============================================================================

for model_name in all_predictions.keys():

    probs = all_predictions[model_name]["probabilities"]

    labels = all_predictions[model_name]["labels"]

    labels_bin = label_binarize(
        labels,
        classes=np.arange(NUM_CLASSES)
    )

    plt.figure(figsize=(8,8))

    for i in range(NUM_CLASSES):

        fpr, tpr, _ = roc_curve(
            labels_bin[:,i],
            probs[:,i]
        )

        roc_auc = auc(fpr,tpr)

        plt.plot(
            fpr,
            tpr,
            label=f"{SELECTED_CLASSES[i]} ({roc_auc:.2f})"
        )

    plt.plot([0,1],[0,1],'k--')

    plt.xlabel("False Positive Rate")

    plt.ylabel("True Positive Rate")

    plt.title(f"{model_name} ROC Curves")

    plt.legend(fontsize=7)

    plt.show()

In [ ]:
# ==============================================================================
# STEP 14: Model Comparison
# ==============================================================================

comparison = pd.DataFrame(evaluation_results).T

comparison = comparison.sort_values(
    by="F1 Score",
    ascending=False
)

comparison

In [ ]:
# ==============================================================================
# STEP 15: Compare All Models
# ==============================================================================

comparison.plot(
    kind='bar',
    figsize=(10,6)
)

plt.title("Performance Comparison")

plt.ylabel("Score")

plt.ylim(0.80,1.02)

plt.grid(True)

plt.show()

In [ ]:
# ==============================================================================
# STEP 16: Save Best Model
# ==============================================================================

best_model_name = comparison.index[0]

best_model = all_trained_models_results[best_model_name]["model"]

torch.save(
    best_model.state_dict(),
    "/content/Best_Tomato_Disease_Model.pth"
)

print(f"Best Model: {best_model_name}")

In [ ]:
# ==============================================================================
# STEP 17: Tomato Disease Prediction Demo
# ==============================================================================

from google.colab import files
from PIL import Image

def predict_image(model, image_path):

    model.eval()

    image = Image.open(image_path).convert("RGB")

    plt.figure(figsize=(5,5))
    plt.imshow(image)
    plt.axis("off")
    plt.title("Uploaded Image")
    plt.show()

    tensor = eval_transforms(image)

    tensor = tensor.unsqueeze(0).to(device)

    with torch.no_grad():

        outputs = model(tensor)

        probabilities = torch.softmax(outputs, dim=1)

        confidence, prediction = torch.max(probabilities,1)

    print("="*60)

    print("Predicted Disease :",
          SELECTED_CLASSES[prediction.item()])

    print("Confidence        :",
          f"{confidence.item()*100:.2f}%")

    print("="*60)

In [ ]:
# ==============================================================================
# STEP 18: Upload Image
# ==============================================================================

uploaded = files.upload()

In [ ]:
# ==============================================================================
# STEP 19: Predict Disease
# ==============================================================================

image_path = list(uploaded.keys())[0]

predict_image(best_model, image_path)

In [ ]:
# ==============================================================================
# STEP 20: Top 5 Predictions
# ==============================================================================

def predict_top5(model, image_path):

    model.eval()

    image = Image.open(image_path).convert("RGB")

    tensor = eval_transforms(image)

    tensor = tensor.unsqueeze(0).to(device)

    with torch.no_grad():

        outputs = model(tensor)

        probabilities = torch.softmax(outputs,dim=1)

    top_prob, top_class = torch.topk(probabilities,5)

    print("\nTop 5 Predictions\n")

    for i in range(5):

        print(
            f"{i+1}. {SELECTED_CLASSES[top_class[0][i]]:<45}"
            f"{top_prob[0][i].item()*100:.2f}%"
        )

In [ ]:
predict_top5(best_model, image_path)